# Predecir precios de productos

## Día 3: Entrenamiento!


In [ ]:
# pip installs

!pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers trl accelerate sentencepiece wandb matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Constantes

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer"
HF_USER = "joanby" # tu nombre de HF va aquí!

# Data

DATASET_NAME = f"{HF_USER}/pricer-data"
# O simplemente usa el que he subido
# DATASET_NAME = "joanby/pricer-data"
MAX_SEQUENCE_LENGTH = 182

# Nombre de la ejecución para guardar el modelo en el Hub

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hiper parámetros de QLoRA

LORA_R = 32 #8
LORA_ALPHA = 64 #16
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1
QUANT_4_BIT = True

# Hiper parámetros para el Entrenamiento

EPOCHS = 3
BATCH_SIZE = 16 #2, 4, 16
GRADIENT_ACCUMULATION_STEPS = 1 #2, 4
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Configuración de Admin

STEPS = 50
SAVE_STEPS = 5000
LOG_TO_WANDB = True

%matplotlib inline

In [ ]:
HUB_MODEL_NAME

'joanby/pricer-2024-12-16_07.57.53'

# Más sobre optimizadores

https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizer-choice

El más común es Adam o AdamW (Adam con decaimiento de peso).
Adam logra una buena convergencia al almacenar el promedio móvil de los gradientes anteriores; sin embargo, agrega una huella de memoria adicional del orden de la cantidad de parámetros del modelo.

### Inicia sesión en HuggingFace y Weights & Biases

Si aún no tienes una cuenta de HuggingFace, visita https://huggingface.co para registrarte y crear un token.

Luego, selecciona los secretos para este cuaderno haciendo clic en el ícono de la llave a la izquierda y agrega un nuevo secreto llamado `HF_TOKEN` con el valor como tu token.

Repite esto para weightsandbiases en https://wandb.ai y agrega un secreto llamado `WANDB_API_KEY`

In [ ]:
# Log in en HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in en Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configuramos Weights & Biases para almacenar la info de nuestro proyecto
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

Generating train split:   0%|          | 0/400000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
test[0]

{'text': "¿Cuánto cuesta esto redondeado al dólar más cercano?\n\nOEM AC Compressor w/A/C Repair Kit For Ford F150 F-150 V8 & Lincoln Mark LT 2007 2008 - BuyAutoParts NEW\nAs one of the world's largest automotive parts suppliers, our parts are trusted every day by mechanics and vehicle owners worldwide. This A/C Compressor and Components Kit is manufactured and tested to the strictest OE standards for unparalleled performance. Built for trouble-free ownership and 100% visually inspected and quality tested, this A/C Compressor and Components Kit is backed by our 100% satisfaction guarantee. Guaranteed Exact Fit for easy installation 100% BRAND NEW, premium ISO/TS 16949 quality - tested to meet or exceed OEM specifications Engineered for superior durability, backed by industry-leading unlimited-mileage warranty Included in this K\n\nPrice is $",
 'price': 374.41}

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

KeyboardInterrupt: 

## Ahora cargue el tokenizador y el modelo

El modelo está "cuantificado": estamos reduciendo la precisión a 4 bits.

In [ ]:
# Elegimos la opción de cuantización adecuada

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [ ]:
# Cargamos el Tokenizer y el Modelo

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/109M [00:00<?, ?B/s]

Memory footprint: 5700.6 MB


# Recopilador de datos

Es importante que nos aseguremos durante el entrenamiento de que no estamos intentando entrenar al modelo para predecir la descripción de los productos, sino solo su precio.

Debemos decirle al entrenador que todo hasta "El precio es $" está ahí para dar contexto al modelo para predecir el siguiente token, pero no es necesario aprenderlo.

El entrenador debe enseñarle al modelo a predecir el token o tokens después de "El precio es $".

Hay una forma complicada de hacer esto configurando máscaras, pero afortunadamente HuggingFace proporciona una clase auxiliar súper simple que se encarga de esto por nosotros.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "Price is $"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# Y AHORA

## Configuramos la configuración para el entrenamiento

Necesitamos crear 2 objetos:

Un objeto LoraConfig con nuestros hiperparámetros para LoRA

Un SFTConfig con nuestros parámetros generales de entrenamiento

In [ ]:
# Primero, especifique los parámetros de configuración para LoRA

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

#A continuación, especifique los parámetros de configuración general para el entrenamiento.

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True
)

# Y ahora, el entrenador de ajuste fino supervisado realizará el ajuste fino
# Dados estos 2 conjuntos de parámetros de configuración

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    tokenizer=tokenizer,
    args=train_parameters,
    data_collator=collator
)

<ipython-input-19-edacc75bc592>:48: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  fine_tuning = SFTTrainer(


Map:   0%|          | 0/400000 [00:00<?, ? examples/s]

In [ ]:
# Fine-tune!
fine_tuning.train()

# Subimos nuestro modelo ajustado a Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Almacenado en el hub: {PROJECT_RUN_NAME}")

Step,Training Loss
50,6.162000
100,5.720400
150,5.589000
200,5.771800
250,5.432600
300,5.736200
350,5.639000
400,5.409600
450,5.652700
500,5.519200


/usr/local/lib/python3.10/dist-packages/trl/trainer/utils.py:169: UserWarning: Could not find response key `Price is $` in the following instance: <|begin_of_text|>¿Cuánto cuesta esto redondeado al dólar más cercano?

Vandoren SM731 BL3 Optimum Series Baritone Saxophone Mouthpiece
The favorite of classical saxophonists. The conception of this series was the fruit of research on both external shape (particularly the beak) and sound, in order to meet the needs of today's saxo- phone players. The'standard' for baritone players Tip Opening - 185 Short Facing Works best with Vandoren Reeds Weight 2.46 ounces, Dimensions 1.75 x 6 x 1.75 inches, Domestic Shipping Currently, item can be shipped only within the U.S. and to APO/FPO addresses. For APO/FPO shipments, please check with the manufacturer regarding warranty and support issues., International Shipping This item can be shipped to select countries outside of the

Price is. This instance will be ignored in loss calculation. Note, if this 

KeyboardInterrupt: 

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()